# 02 · Volume, intensidade e relação pace × FC

Observação do lado do **treino**: o que foi feito e com que intensidade. Usa `activities` (uma linha
por treino) e `hr_zones` (uma linha por atividade e zona).

**Perguntas deste notebook**

1. Quanto e o que se treina por semana?
2. A distribuição de intensidade das corridas segue o modelo polarizado (80% fácil / 20% forte)?
3. Como pace e frequência cardíaca se relacionam — e essa relação melhorou ao longo dos meses?

Sono e HRV não entram aqui. O que depende dos dois lados está em `03_load_recovery`.

In [ ]:
from config import IMG_DIR
from processing.datasets import join_activities, load_activities, load_hr_zones
from processing.features import auditar_nulos
from processing.plots import (
    FAIXAS_CORES,
    ZONAS_CORES,
    ZONAS_LABELS,
    ZONAS_ORDEM,
    colapsar_faixas,
    eixo_pace,
)

import matplotlib.pyplot as plt
import numpy as np

activities = load_activities()
hr_zones = load_hr_zones()

runs = activities[activities["sport"] == "Run"].copy()

print(f"{len(activities)} atividades | {len(runs)} corridas | {activities['date'].min().date()} a {activities['date'].max().date()}")
activities.head()

## 1. Volume semanal de corrida

Quilometragem por semana, com média móvel de 4 semanas. A média móvel é o que mostra progressão de
carga; a barra isolada mistura bloco de treino com semana de viagem.

O agrupamento usa `comeco_semana` (a segunda-feira) em vez do número da semana, porque no eixo de
data as lacunas aparecem como lacunas — agrupar pelo número sequencial encostaria semanas distantes
uma na outra e esconderia o intervalo sem treino.

In [ ]:
volume = runs.groupby("comeco_semana").agg(
    km=("distance_km", "sum"),
    treinos=("activity_id", "count"),
    minutos=("duration_minutes", "sum"),
)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(volume.index, volume["km"], width=5, color="#3498db", alpha=0.85, label="km na semana")
ax.plot(
    volume.index,
    volume["km"].rolling(4, min_periods=1).mean(),
    color="#e74c3c", linewidth=2, label="média 4 semanas",
)

ax.set_ylabel("km")
ax.set_title("Volume semanal de corrida")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

volume.tail(8).round(1)

## 2. Zonas de FC — formato longo

`hr_zones` está em formato longo: uma linha por combinação de atividade e zona. Uma corrida que passou
por quatro zonas ocupa quatro linhas. É o formato que faz o `groupby` da próxima seção funcionar
direto — em formato largo seria preciso derreter a tabela antes.

Dois filtros ficam de pé para o resto do notebook:

- **só corridas** — musculação e caminhada têm dinâmica de FC diferente e distorceriam a leitura;
- **sem `zone_0`** — é o tempo abaixo de Z1: pausa, semáforo, descanso entre séries. Não é treino.

In [ ]:
runs_hr = hr_zones[(hr_zones["sport"] == "Run") & (hr_zones["zone"] != "zone_0")].copy()

print(f"{len(runs_hr)} linhas | {runs_hr['activity_id'].nunique()} corridas com zonas registradas")
runs_hr.head()

## 3. Distribuição semanal do tempo por zona

Minutos por (semana, zona) convertidos em **percentual dentro de cada semana**. O percentual é o que
importa aqui: o volume total varia muito de semana para semana, e comparar minutos absolutos
misturaria "treinou pouco" com "treinou fácil".

O `reindex` explícito de Z1 a Z5 garante que uma zona sem registro na semana vire zero em vez de sumir
da área empilhada. A linha tracejada em 80% é a referência do modelo polarizado — idealmente Z1+Z2
ficam acima dela.

In [ ]:
semana_zona = runs_hr.groupby(["semana", "zone"])["minutes"].sum().reset_index()
semana_zona["pct"] = (
    semana_zona["minutes"] / semana_zona.groupby("semana")["minutes"].transform("sum") * 100
)

zonas_pct = (
    semana_zona.pivot(index="semana", columns="zone", values="pct")
    .reindex(columns=ZONAS_ORDEM, fill_value=0)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.stackplot(
    zonas_pct.index.astype(str),
    *[zonas_pct[zona] for zona in ZONAS_ORDEM],
    labels=ZONAS_LABELS,
    colors=ZONAS_CORES,
    alpha=0.85,
)

ax.axhline(80, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(0, 81, "80% (meta Z1+Z2 no modelo polarizado)", fontsize=9, color="gray")
ax.set_ylim(0, 100)
ax.set_xlabel("Semana")
ax.set_ylabel("% do tempo de corrida")
ax.set_title("Distribuição semanal do tempo de corrida por zona de FC")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.show()

## 4. Leitura polarizada: três faixas

Cinco zonas são detalhe demais para ver o padrão. `colapsar_faixas()` soma as mesmas colunas nas três
faixas do modelo polarizado:

| Faixa | Zonas | O que representa |
|---|---|---|
| Baixa intensidade | Z1 + Z2 | Volume aeróbico, deveria dominar a semana |
| Gray zone | Z3 | Forte demais para recuperar, leve demais para gerar adaptação |
| Alta intensidade | Z4 + Z5 | Limiar e VO2max, o estímulo forte |

Semana com faixa laranja grande é o sinal clássico de treino "sempre médio": cansa como treino forte e
rende como treino fácil.

In [ ]:
faixas = colapsar_faixas(zonas_pct)

fig, ax = plt.subplots(figsize=(14, 6))
ax.stackplot(
    faixas.index.astype(str),
    *[faixas[col] for col in faixas.columns],
    labels=faixas.columns,
    colors=FAIXAS_CORES,
    alpha=0.85,
)

ax.axhline(80, color="green", linestyle="--", linewidth=1, alpha=0.7)
ax.text(0, 81, "80% — alvo Z1+Z2", fontsize=9, color="green")
ax.set_ylim(0, 100)
ax.set_xlabel("Semana")
ax.set_ylabel("% do tempo de corrida")
ax.set_title("Distribuição polarizada do treino de corrida")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.show()

print("Média do período (% do tempo de corrida):")
print(faixas.mean().round(1).to_string())
print(f"\nSemanas com Z1+Z2 acima de 80%: {(faixas.iloc[:, 0] >= 80).sum()} de {len(faixas)}")

## 5. Zonas cruzadas com os dados da atividade

A tabela de zonas só tem `activity_id`, data e esporte. Para analisar zona junto com distância, pace ou
TSS é preciso trazer as colunas de `activities` — é o que `join_activities()` faz, com
`validate="many_to_one"`: se algum `activity_id` estivesse duplicado do lado direito, o merge
multiplicaria linhas em silêncio e todo agregado seguinte sairia inflado. Com o `validate`, o pandas
levanta erro em vez de deixar passar.

Na auditoria de nulos que segue, colunas 100% vazias são as de natação e musculação — esperado depois
de filtrar só corridas. As que ficam entre 15% e 25% são treinos que vieram só do Garmin, sem par no
Strava.

In [ ]:
runs_hr_full = join_activities(runs_hr)

print(f"{runs_hr.shape[1]} colunas antes do join, {runs_hr_full.shape[1]} depois")
auditar_nulos(runs_hr_full).head(15)

## 6. Pace × FC nas corridas de base aeróbica

Terceira pergunta do notebook. A ideia: em corridas de mesma intensidade, **correr mais rápido com a
mesma FC significa que a aptidão aeróbica melhorou**.

O filtro isola `AEROBIC_BASE` para comparar coisas comparáveis — um tiro de VO2max e uma rodagem leve
têm relações pace/FC completamente diferentes e, misturados, viram nuvem sem sentido.

- **Cor por mês** (viridis) mostra a evolução temporal sem agrupar em faixas: se os pontos claros
  (meses recentes) ficarem à esquerda e abaixo dos escuros, houve ganho de forma.
- **Eixo Y invertido e em MM:SS** (`eixo_pace`), porque pace menor é melhor — assim "para cima"
  continua significando "melhor".
- **Reta tracejada**: regressão linear simples, só como referência visual da tendência.

In [ ]:
base_runs = runs[
    (runs["training_effect_label"] == "AEROBIC_BASE")
    & runs["avg_heart_rate"].notna()
    & runs["avg_pace_sec_per_km"].notna()
].copy()

base_runs["mes"] = base_runs["date"].dt.to_period("M")

fig, ax = plt.subplots(figsize=(12, 7))

meses = sorted(base_runs["mes"].unique())
cmap = plt.cm.viridis

for i, mes in enumerate(meses):
    sub = base_runs[base_runs["mes"] == mes]
    ax.scatter(
        sub["avg_heart_rate"],
        sub["avg_pace_sec_per_km"],
        color=cmap(i / max(len(meses) - 1, 1)),
        label=str(mes),
        s=60, alpha=0.75, edgecolors="black", linewidth=0.5,
    )

coef = np.polyfit(base_runs["avg_heart_rate"], base_runs["avg_pace_sec_per_km"], 1)
x_linha = np.linspace(base_runs["avg_heart_rate"].min(), base_runs["avg_heart_rate"].max(), 100)
ax.plot(x_linha, np.polyval(coef, x_linha), "--", color="red", alpha=0.5, label="tendência")

eixo_pace(ax)
ax.set_xlabel("FC média (bpm)")
ax.set_ylabel("Pace médio (min/km)")
ax.set_title(f"Pace × FC — corridas de base aeróbica (n={len(base_runs)})")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, title="Mês")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Mesma relação, todas as corridas — e o baseline

O recorte anterior é estreito demais para uma reta confiável. Aqui o filtro se amplia para todas as
corridas com FC, pace e tipo de treino válidos, e a cor passa a codificar o **tipo de treino** em vez
do mês.

O que se espera ver: os grupos se organizando em diagonal — Recovery no canto de FC baixa e pace
lento, Lactate Threshold no extremo oposto. Grupos sobrepostos indicam que as zonas do relógio não
estão calibradas para a FC máxima real.

A inclinação da reta (segundos por km a cada batimento) é o número a guardar: refazendo o mesmo
cálculo daqui a alguns meses, a mudança nessa inclinação é uma medida objetiva de progresso. O gráfico
é salvo em `IMG_DIR` exatamente para essa comparação.

In [ ]:
all_runs = runs[
    runs["avg_heart_rate"].notna()
    & runs["avg_pace_sec_per_km"].notna()
    & runs["training_effect_label"].notna()
].copy()

print(f"Corridas no recorte: {len(all_runs)} de {len(runs)}")
print(all_runs["training_effect_label"].value_counts().to_string())

tipos = {
    "RECOVERY": {"color": "#2ecc71", "marker": "o", "label": "Recovery"},
    "AEROBIC_BASE": {"color": "#3498db", "marker": "o", "label": "Aerobic Base"},
    "TEMPO": {"color": "#f39c12", "marker": "s", "label": "Tempo (Z3)"},
    "LACTATE_THRESHOLD": {"color": "#e74c3c", "marker": "^", "label": "Lactate Threshold"},
}

fig, ax = plt.subplots(figsize=(13, 7))

for tipo, cfg in tipos.items():
    sub = all_runs[all_runs["training_effect_label"] == tipo]
    if sub.empty:
        continue
    ax.scatter(
        sub["avg_heart_rate"],
        sub["avg_pace_sec_per_km"],
        color=cfg["color"], marker=cfg["marker"], label=f"{cfg['label']} (n={len(sub)})",
        s=60, alpha=0.65, edgecolors="black", linewidth=0.4,
    )

coef = np.polyfit(all_runs["avg_heart_rate"], all_runs["avg_pace_sec_per_km"], 1)
x_linha = np.linspace(all_runs["avg_heart_rate"].min(), all_runs["avg_heart_rate"].max(), 100)
ax.plot(x_linha, np.polyval(coef, x_linha), "--", color="black", alpha=0.4, linewidth=1.5,
        label="tendência linear (todas)")

eixo_pace(ax)
ax.set_xlabel("FC média (bpm)")
ax.set_ylabel("Pace médio (min/km)")
ax.set_title(f"Relação pace × FC — todas as corridas (n={len(all_runs)})")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, title="Tipo de treino")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / "pace_fc_baseline_2026.png", dpi=300)
plt.show()

print(f"Cada +1 bpm de FC corresponde a pace {abs(coef[0]):.1f} s/km mais rápido")

## Observações

_A preencher a cada atualização._

- O recorte com tipo de treino cobre 45 das 126 corridas — `training_effect_label` só vem do Garmin, e
  boa parte das atividades antigas veio só do Strava. Conclusão por tipo de treino vale para esse
  subconjunto, não para o histórico inteiro.
- Anotar aqui: inclinação s/km por bpm da rodada atual (para comparar com a próxima), semanas fora do
  padrão polarizado e o que explicou cada uma.